In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama
from typing import TypedDict,Literal
from pydantic import BaseModel,Field

In [2]:
model=ChatOllama(model="qwen3:4b-instruct")

In [3]:
class sentimentschema(BaseModel):
    sentiment:Literal["positive","negative"]=Field(description="Sentiment of the review")

In [15]:
class diagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [16]:
structured_model=model.with_structured_output(sentimentschema)
structured_model2=model.with_structured_output(diagnosisSchema)

In [ ]:
prompt="What is the sentiment of the review the software is not good as much I expected"
structured_model.invoke(prompt)

sentimentschema(sentiment='negative')

In [7]:
class reviewstate(TypedDict):
    review:str
    sentiment:Literal["positive","negative"]
    diagnosis:dict
    response:str

In [ ]:
def find_sentiment(state:reviewstate):
    prompt=f'What is the sentiment of the review \n {state["review"]}'

    sentiment=structured_model.invoke(prompt)

    return {"sentiment":sentiment}.sentiment

In [13]:
def positive_response(state:reviewstate):
       prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\n
Also, kindly ask the user to leave feedback on our website."""
       response=model.invoke(prompt).content

       return {"response":response}

In [19]:
def run_diagnosis(state:reviewstate):
    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n"
    "Return issue_type, tone, and urgency.
"""
    response=structured_model2.invoke(prompt)

    return {"diagnosis":response.model_dump()}

In [20]:
sample_input = {
    "review": "I am extremely disappointed with this purchase. The product arrived three days late, and the packaging was completely damaged. When I finally opened it, the device wouldn't even turn on. I tried reaching out to customer support twice, and no one has bothered to reply yet. Total waste of money."
}

result=run_diagnosis(sample_input)
print(result)

{'diagnosis': {'issue_type': 'Performance', 'tone': 'angry', 'urgency': 'high'}}


In [ ]:
def negative_response(state:reviewstate):
    diagnosis=state["diagnosis"]
    prompt=f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response=model.invoke(prompt).content

    return {"response":response}

In [ ]:
graph=StateGraph(reviewstate)

graph.add_node("find_sentiment",find_sentiment)
graph.add_node("positive_response",positive_response)
graph.add_node("negative_response",negative_response)
graph.add_node("run_diagnosis",run_diagnosis)